In [15]:
import pandas as pd
from catboost import CatBoostClassifier

In [16]:
train_path = 'datasets/train.csv'
test_path = 'datasets/test.csv'

train_df = pd.read_csv(train_path)
test_df = pd.read_csv(test_path)

print('train: ', train_df.shape)
print('test: ', test_df.shape)
print(train_df['Heart Disease'].value_counts())

train:  (630000, 15)
test:  (270000, 14)
Heart Disease
Absence     347546
Presence    282454
Name: count, dtype: int64


### Подготовка признаков

In [17]:
target_col = 'Heart Disease'
y = train_df[target_col].map({"Absence": 0, "Presence": 1})
X = train_df.drop(columns=[target_col])
X_test = test_df.copy()

In [18]:
cat_cols = [
    "Sex", "Chest pain type",
    "FBS over 120", "EKG results",
    "Exercise angina", "Slope of ST",
    "Number of vessels fluro", "Thallium",
]

In [19]:
# задаю индексы категориальных колонок для catboost
cat_features = [X.columns.get_loc(c) for c in cat_cols]
print("X:", X.shape, "y:", y.shape, "X_test:", X_test.shape)
print("cat feature idx:", cat_features)

X: (630000, 14) y: (630000,) X_test: (270000, 14)
cat feature idx: [2, 3, 6, 7, 9, 11, 12, 13]


### Кросс-валидация (CV)
(опционально, может быть долго на 630k строках)


In [20]:
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (f1_score, fbeta_score, roc_auc_score, average_precision_score)

skf = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
cv_rows = []

# небольшая сетка параметров для быстрого тюнинга
param_grid = [
    {'depth': 6, 'learning_rate': 0.05, 'l2_leaf_reg': 3},
    {'depth': 8, 'learning_rate': 0.05, 'l2_leaf_reg': 5},
    {'depth': 6, 'learning_rate': 0.1, 'l2_leaf_reg': 3},
]

for params in param_grid:
    for fold, (tr_idx, va_idx) in enumerate(skf.split(X, y), 1):
        X_tr, X_va = X.iloc[tr_idx], X.iloc[va_idx]
        y_tr, y_va = y.iloc[tr_idx], y.iloc[va_idx]

        cv_model = CatBoostClassifier(
            iterations=1500,
            depth=params['depth'],
            learning_rate=params['learning_rate'],
            l2_leaf_reg=params['l2_leaf_reg'],
            loss_function='Logloss', eval_metric='AUC',
            random_seed=42, verbose=False,
            od_type='Iter', od_wait=100
        )
        cv_model.fit(
            X_tr, y_tr,
            cat_features=cat_features,
            eval_set=(X_va, y_va),
            use_best_model=True
        )

        va_proba = cv_model.predict_proba(X_va)[:, 1]
        va_pred = (va_proba >= 0.5).astype(int)

        cv_rows.append({
            'depth': params['depth'],
            'learning_rate': params['learning_rate'],
            'l2_leaf_reg': params['l2_leaf_reg'],
            'fold': fold,
            'f1': f1_score(y_va, va_pred),
            'f2': fbeta_score(y_va, va_pred, beta=2),
            'roc_auc': roc_auc_score(y_va, va_proba),
            'pr_auc': average_precision_score(y_va, va_proba),
        })

cv_df = pd.DataFrame(cv_rows)
print(cv_df)
print('mean by params:')
print(cv_df.groupby(['depth','learning_rate','l2_leaf_reg']).mean(numeric_only=True))


   depth  learning_rate  l2_leaf_reg  fold        f1        f2   roc_auc  \
0      6           0.05            3     1  0.875244  0.871544  0.955348   
1      6           0.05            3     2  0.875023  0.869659  0.955158   
2      6           0.05            3     3  0.874426  0.869506  0.955581   
3      8           0.05            5     1  0.874775  0.871243  0.955118   
4      8           0.05            5     2  0.874594  0.869187  0.955008   
5      8           0.05            5     3  0.874217  0.869392  0.955384   
6      6           0.10            3     1  0.875107  0.871259  0.955316   
7      6           0.10            3     2  0.874834  0.869398  0.955140   
8      6           0.10            3     3  0.874420  0.869543  0.955553   

     pr_auc  
0  0.948519  
1  0.948504  
2  0.949140  
3  0.948223  
4  0.948326  
5  0.948885  
6  0.948473  
7  0.948498  
8  0.949112  
mean by params:
                                 fold        f1        f2   roc_auc    pr_auc
depth

### Финальная модель по лучшим параметрам CV


In [21]:
# выбираем лучшие параметры по среднему roc_auc
mean_df = cv_df.groupby(['depth','learning_rate','l2_leaf_reg']).mean(numeric_only=True)
best_params = mean_df.sort_values('roc_auc', ascending=False).iloc[0]
best_depth = int(best_params.name[0])
best_lr = float(best_params.name[1])
best_l2 = float(best_params.name[2])
print('best params:', {'depth': best_depth, 'learning_rate': best_lr, 'l2_leaf_reg': best_l2})

final_model = CatBoostClassifier(
    iterations=3000,
    depth=best_depth,
    learning_rate=best_lr,
    l2_leaf_reg=best_l2,
    loss_function='Logloss', eval_metric='AUC',
    random_seed=42, verbose=200,
    od_type='Iter', od_wait=200
)
final_model.fit(
    X, y,
    cat_features=cat_features
)


best params: {'depth': 6, 'learning_rate': 0.05, 'l2_leaf_reg': 3.0}
0:	total: 104ms	remaining: 5m 11s
200:	total: 20.1s	remaining: 4m 39s
400:	total: 42.7s	remaining: 4m 36s
600:	total: 1m 5s	remaining: 4m 21s
800:	total: 1m 28s	remaining: 4m 2s
1000:	total: 1m 50s	remaining: 3m 41s
1200:	total: 2m 13s	remaining: 3m 19s
1400:	total: 2m 36s	remaining: 2m 58s
1600:	total: 2m 59s	remaining: 2m 36s
1800:	total: 3m 22s	remaining: 2m 15s
2000:	total: 3m 46s	remaining: 1m 53s
2200:	total: 4m 10s	remaining: 1m 30s
2400:	total: 4m 33s	remaining: 1m 8s
2600:	total: 4m 57s	remaining: 45.6s
2800:	total: 5m 21s	remaining: 22.8s
2999:	total: 5m 45s	remaining: 0us


### Разбиваю данные на выборки

In [22]:
from sklearn.model_selection import train_test_split
X_train, X_val, y_train, y_val = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print(X_train.shape, X_val.shape)

(504000, 14) (126000, 14)


### Обучаю модель catboost


In [23]:
from catboost import CatBoostClassifier
from sklearn.metrics import (
    f1_score, fbeta_score,
    roc_auc_score, average_precision_score
)

In [24]:
model = CatBoostClassifier(
    iterations=2000, depth=6, learning_rate=0.05,
    loss_function='Logloss', eval_metric='AUC',
    random_seed=42, verbose=200,
    od_type='Iter', od_wait=100
)


In [25]:
model.fit(
    X_train, y_train,
    cat_features=cat_features,
    eval_set=(X_val, y_val),
    use_best_model=True
)

0:	test: 0.9359527	best: 0.9359527 (0)	total: 84.4ms	remaining: 2m 48s
200:	test: 0.9551929	best: 0.9551929 (200)	total: 18s	remaining: 2m 41s
400:	test: 0.9557629	best: 0.9557629 (400)	total: 37.2s	remaining: 2m 28s
600:	test: 0.9559736	best: 0.9559736 (600)	total: 56.9s	remaining: 2m 12s
800:	test: 0.9560979	best: 0.9560979 (800)	total: 1m 16s	remaining: 1m 54s
1000:	test: 0.9561492	best: 0.9561492 (1000)	total: 1m 35s	remaining: 1m 35s
1200:	test: 0.9561754	best: 0.9561765 (1186)	total: 1m 56s	remaining: 1m 17s
1400:	test: 0.9562007	best: 0.9562020 (1392)	total: 2m 16s	remaining: 58.3s
1600:	test: 0.9562052	best: 0.9562087 (1555)	total: 2m 37s	remaining: 39.1s
Stopped by overfitting detector  (100 iterations wait)

bestTest = 0.956208684
bestIteration = 1555

Shrink model to first 1556 iterations.


In [26]:
val_proba = model.predict_proba(X_val)[:, 1]
val_pred = (val_proba >= 0.5).astype(int)

In [27]:
metrics = {
    "f1": f1_score(y_val, val_pred),
    "f2": fbeta_score(y_val, val_pred, beta=2),
    "roc_auc": roc_auc_score(y_val, val_proba),
    "pr_auc": average_precision_score(y_val, val_proba),
}

#### Делаем сабмишн

In [28]:
submit_model = final_model if 'final_model' in globals() else model
test_proba = submit_model.predict_proba(X_test)[:, 1]

submission = pd.DataFrame({
    'id': X_test['id'],
    'Heart Disease': test_proba
})

submission.to_csv('submissions/cv_best.csv', index=False)
print(submission.head())


       id  Heart Disease
0  630000       0.953466
1  630001       0.006830
2  630002       0.992276
3  630003       0.002866
4  630004       0.218514
